# Query 5: CASE, Functions, and NULLs

Run `ddl.ipynb` and `dml.ipynb` first. This notebook labels values, builds conditional summaries, and handles missing or blank text.

In [ ]:
%load_ext sql
%sql sqlite:///food-analysis/food_analysis.db

## Group order statuses into readable labels

`CASE` evaluates conditions in order and returns the first matching result. `ELSE` provides a label for any other value.

In [ ]:
%%sql
SELECT order_id, status,
       CASE
           WHEN status = 'delivered' THEN 'Complete'
           WHEN status = 'cancelled' THEN 'Cancelled'
           WHEN status IN ('pending', 'returned') THEN 'Needs attention'
           ELSE 'Other'
       END AS status_group
FROM orders
ORDER BY order_id
LIMIT 12;

## Compare delivery outcomes by partner

Putting a `CASE` expression inside an aggregate lets you count or sum only rows that meet a condition.

In [ ]:
%%sql
SELECT partner,
       COUNT(*) AS all_orders,
       SUM(CASE WHEN status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
       ROUND(SUM(CASE WHEN status = 'delivered' THEN order_amount ELSE 0 END), 2) AS delivered_amount
FROM orders
GROUP BY partner
ORDER BY delivered_amount DESC;

## Treat missing and blank areas consistently

`TRIM` removes surrounding spaces, `NULLIF` turns an empty string into `NULL`, and `COALESCE` supplies a display value when the result is `NULL`.

In [ ]:
%%sql
SELECT customer_id,
       customer_name,
       COALESCE(NULLIF(TRIM(area), ''), 'Unknown area') AS display_area
FROM customers
ORDER BY customer_id
LIMIT 12;

## Use SQLite text and date functions

SQLite provides functions such as `UPPER` and `strftime`. `strftime('%Y-%m', ...)` extracts a year-month label from the stored ISO date text.

In [ ]:
%%sql
SELECT UPPER(partner) AS partner,
       strftime('%Y-%m', order_placed_at) AS order_month,
       COUNT(*) AS order_count
FROM orders
GROUP BY UPPER(partner), strftime('%Y-%m', order_placed_at)
ORDER BY order_month DESC, partner;